# Descriptive, Analytical and Policy Questions

**DS4DH · Module 01 — Framing the Right Question**

*Technique:* Question typology — matching a question to the method that can answer it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/01a_question_typology.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Three questions about the same column, each needing a different method and each
licensing a different kind of claim:

| Type | Question | Method | Claim you may make |
|---|---|---|---|
| **Descriptive** | What is renter STIR in Toronto? | mean, median | "It is X." |
| **Analytical** | Is renter STIR higher for immigrants, and is that difference real? | grouped comparison, hypothesis test | "There is a gap of X, unlikely to be chance." |
| **Policy** | Should Toronto expand rent supplements? | the above, plus a modifiable factor and an institution that can act | "The evidence supports X, subject to Y." |

The failure this notebook is designed to prevent is answering a descriptive
question and reporting it as if it were a policy one.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

## Level 1 — Descriptive

A descriptive question has an arithmetic answer. It cannot be wrong about
causation because it makes no causal claim at all.

In [ ]:
d = base.dropna(subset=['Renter'])
print('Renter STIR by city — descriptive only')
print()
print(f'{"City":<12}{"n":>5}{"mean":>8}{"median":>8}{"min":>8}{"max":>8}')
print('-' * 49)
for city in CITIES:
    s = d[d['cma'] == city]['Renter']
    print(f'{city:<12}{len(s):>5}{s.mean():>8.1f}{s.median():>8.1f}'
          f'{s.min():>8.1f}{s.max():>8.1f}')

## Level 2 — Analytical

An analytical question compares. It needs a second group, and it needs some
account of whether the difference could be noise.

Here the comparison is between immigrant and non-immigrant renters *within the
same CSD*, which controls for the place automatically.

In [ ]:
csd = df.dropna(subset=['csd_code'])
imm = csd[csd['immigrant_status'] == 'Immigrant'][['csd_code', 'cma', 'Renter']]
nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']]
pair = imm.merge(nim, on='csd_code', suffixes=('_imm', '_nim')).dropna()
pair = pair[pair['cma'].isin(CITIES)]
pair['gap'] = pair['Renter_imm'] - pair['Renter_nim']

print(f'{len(pair)} CSDs where both groups are reported')
print()
print(f'{"City":<12}{"n":>5}{"mean gap (pp)":>16}')
print('-' * 33)
for city in CITIES:
    g = pair[pair['cma'] == city]['gap']
    print(f'{city:<12}{len(g):>5}{g.mean():>+16.2f}')
print()
print('Note the sign. One of these is not like the others.')

### 🔧 Your turn 1

One city has a mean gap with the opposite sign to the other three.

Before reading on: is that enough to say immigrant renters in that city are
better off? What would you need to know first? (Module 04 answers this
properly; the point here is to notice that the question has changed.)

## Level 3 — Policy

A policy question needs three things a statistic cannot supply on its own:

1. the finding is **real** (survives a significance test and has a usable effect size)
2. it points at a **modifiable** factor — something an institution can change
3. there is an **institution** whose remit covers that factor

Fail any one and you have an interesting finding, not a recommendation. The cell
below does not compute a recommendation — it computes the *scale* that a
recommendation would have to be proportionate to.

In [ ]:
# Translating a percentage-point gap into money, which is what a
# housing agency actually budgets in.
sub = base.dropna(subset=['Renter', 'rent_income'])
med_income = sub['rent_income'].median()

print(f'Median renter household income across the 4 CMAs: ${med_income:,.0f}')
print()
for pp in [1, 3, 5]:
    print(f'  a {pp}pp difference in STIR = ${med_income * pp / 100:>8,.0f} per year')
print()
print('This is the arithmetic that turns "3 percentage points" into a number')
print('a policy reader can weigh against a programme budget.')

### 🔧 Your turn 2

Change `med_income` to use `.mean()` instead of `.median()` and re-run.

The dollar figures move. Which one belongs in a policy brief, and why? (There is
a defensible answer either way — what matters is that you can say which you used.)

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** No. A mean gap in the opposite direction tells you the average
sign in a sample of 13 CSDs, and nothing about whether that average would hold
in another sample. It could be one unusual municipality pulling the mean. You
need a test of whether the difference is distinguishable from zero, and an
effect size to say whether it is large enough to matter. Edmonton turns out to
be the one finding in this dataset that survives both — see notebook 04b.

**Your turn 2.** The mean is pulled upward by a few very high-income CSDs, so it
overstates the typical renter household. For a policy brief about affordability
pressure, the median is the safer choice: it describes the household in the
middle. The mean is defensible if you are budgeting a programme in aggregate,
since total cost depends on the sum, not the middle. The failure is using
whichever produced the more dramatic number without saying so.

</details>

## Where this stops

You have three questions and three methods, and you can tell them apart. What
you cannot yet do is notice what the dataset never recorded — which is the
subject of the next notebook, and the more common way analyses go wrong.